# 1. MIMIC-IV hosp-only: raw tables → prediction cohort

Run top to bottom. All code is included. **Only the seven `hosp` tables listed
below are read.** Insulin exposure comes from `emar`; glucose, including labels
and baseline measurements, comes from `labevents`. No ICU tables or derived
ICU features are used. This source restriction does not exclude patients who
happened to receive intensive care.

Predict documented glucose <70 mg/dL within 24h after documented insulin.
Select non-overlapping windows before looking at outcomes; keep each patient
in one split. Require sufficient monitoring for both positive and negative labels.
Intermittent lab monitoring may miss transient lows and restrict the usable cohort.

The supported T2DM phenotype is retrospective and not expert-validated. Evidence
may occur after prediction; it is never a predictor. The target is an association
following insulin, not proof of drug causation. Earlier mixed-source performance
numbers do not apply to this cohort.

`READY_DIR` is the complete handoff to notebook 2. Checkpoints in `CACHE_DIR`
support restarting long scans. Use a new cache when changing data or definitions.

## Setup

Install before importing. Restart if Colab requests it.

In [ ]:
%pip -q install "numpy==2.1.3" "pandas==2.2.3" "pyarrow==19.0.0"

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # Local Jupyter: use local paths in the configuration cell below.

## Paths and runtime

Point `HOSP_DIR` directly to the folder containing the compressed hosp CSV files. Each teammate mounts their own Drive. [Colab Drive guide](https://colab.research.google.com/notebooks/io.ipynb).

In [ ]:
import os
from pathlib import Path

for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "2"
HOSP_DIR = Path("/content/drive/MyDrive/mimiciv/3.1/hosp")
CACHE_DIR = Path("/content/drive/MyDrive/BME790/cache_hosp_v1")
READY_DIR = Path("/content/drive/MyDrive/BME790/preprocessed_hosp")
EXISTING_SPLITS = None  # Optional earlier .csv.gz/.parquet patient split manifest.
CHUNK_SIZE = 1_000_000
N = 32  # Admission-hash partitions bound memory use.

## Validate inputs and cache settings

All paths are relative to `HOSP_DIR`. The source signature prevents accidental reuse of an earlier mixed-source cache.

In [ ]:
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ROOT, OUT = HOSP_DIR, CACHE_DIR
H = pd.Timedelta(hours=1)
KEY = ["subject_id", "hadm_id"]
required = [
    "patients",
    "admissions",
    "diagnoses_icd",
    "d_icd_diagnoses",
    "prescriptions",
    "labevents",
    "emar",
]
missing = [name for name in required if not (ROOT / f"{name}.csv.gz").exists()]
if missing:
    raise FileNotFoundError("Missing hosp tables: " + ", ".join(missing))
OUT.mkdir(parents=True, exist_ok=True)
READY_DIR.mkdir(parents=True, exist_ok=True)
split_hash = None
if EXISTING_SPLITS is not None:
    EXISTING_SPLITS = Path(EXISTING_SPLITS)
    split_hash = hashlib.sha256(EXISTING_SPLITS.read_bytes()).hexdigest()
signature = {
    "protocol": "hosp_only_t2dm_v1",
    "source_modules": ["hosp"],
    "hosp_dir": str(ROOT.resolve()),
    "partitions": N,
    "split_manifest_sha256": split_hash,
}
manifest = OUT / "notebook_config.json"
if manifest.exists() and json.loads(manifest.read_text()) != signature:
    raise ValueError("Cache settings changed. Use a new CACHE_DIR.")
if not manifest.exists() and any(OUT.iterdir()):
    raise ValueError("Unverified existing cache. Use an empty CACHE_DIR.")
manifest.write_text(json.dumps(signature, indent=2))


def log(message):
    print(message, flush=True)

## Clinical definitions

Prior diagnosis groups use earlier completed admissions only. Medication classes use actual eMAR administration names. Current labs remain continuous predictors; abnormal values can influence the model without using future test results.

In [ ]:
MED_CLASSES = {
    "insulin": r"insulin",
    "sulfonylurea": r"glipi[z]ide|glyburide|glimepiride|tolbutamide",
    "other_diabetes_drugs": r"metformin|sitagliptin|linagliptin|saxagliptin|repaglinide|nateglinide|liraglutide|semaglutide|dulaglutide|exenatide|empagliflozin|dapagliflozin|canagliflozin|pioglitazone",
    "steroid": r"predniso|prednisone|methylpred|hydrocortisone|dexamethasone|cortisone",
    "beta_blocker": r"metoprolol|atenolol|carvedilol|propranolol|labetalol|esmolol|bisoprolol|nadolol",
    "ace_arb": r"lisinopril|enalapril|captopril|ramipril|losartan|valsartan|irbesartan|candesartan",
    "diuretic": r"furosemide|bumetanide|torsemide|hydrochlorothiazide|chlorothiazide|metolazone|spironolactone",
    "fluoroquinolone": r"ciprofloxacin|levofloxacin|moxifloxacin",
    "other_antibiotic": r"cef[a-z]+|vancomycin|piperacillin|tazobactam|meropenem|ampicillin|amoxicillin|trimethoprim|sulfamethoxazole|metronidazole|linezolid|doxycycline|azithromycin",
    "antifungal": r"fluconazole|voriconazole|posaconazole|micafungin|caspofungin",
    "opioid": r"morphine|hydromorphone|oxycodone|fentanyl|methadone|tramadol|codeine",
    "sedative": r"lorazepam|diazepam|midazolam|propofol|dexmedetomidine|clonazepam",
    "antipsychotic": r"haloperidol|quetiapine|olanzapine|risperidone|aripiprazole",
    "anticoagulant": r"heparin|enoxaparin|warfarin|apixaban|rivaroxaban|bivalirudin",
    "vasopressor": r"norepinephrine|epinephrine|vasopressin|phenylephrine|dopamine|dobutamine",
    "dextrose_glucagon": r"dextrose|glucagon|glucose gel",
    "potassium": r"potassium",
    "nutrition": r"parenteral nutrition|amino acid|lipid emulsion|enteral|tube feed",
}

HISTORY_PATTERNS = {
    "diabetes": r"diabet",
    "hypoglycemia": r"hypoglyc",
    "kidney_disease": r"kidney|renal|dialysis",
    "liver_disease": r"liver|hepatic|hepatitis|cirrhosis",
    "heart_failure": r"heart failure",
    "vascular_disease": r"ischemic|ischaemic|atherosclero|infarction|peripheral vascular",
    "hypertension": r"hypertens",
    "infection_sepsis": r"sepsis|septic|bacteremia|pneumonia|infection",
    "cancer": r"malignan|neoplasm|carcinoma|leukemia|lymphoma|metastatic",
    "nutrition_weight": r"malnutrition|cachexia|underweight|obesity|weight loss|anorexia",
    "adrenal_endocrine": r"adrenal|pituitary|thyroid",
    "neurologic": r"dementia|encephalop|epilep|cerebral|stroke",
    "respiratory": r"respiratory failure|pulmonary|asthma|emphysema",
    "alcohol_substance": r"alcohol|drug dependence|substance abuse",
    "vomiting_diarrhea": r"vomit|diarrhea|diarrhoea",
    "altered_mental_status": r"altered mental|confusion|syncope|coma|dizziness",
    "anemia": r"anemia|anaemia",
    "electrolyte": r"hyperkal|hypokal|hyponatr|hypernatr|acidosis|alkalosis",
}

RX_PATTERN = r"acetohexamide|tolazamide|chlorpropamide|glipizide|glyburide|glimepiride|repaglinide|nateglinide|metformin|rosiglitazone|pioglitazone|troglitazone|acarbose|miglitol|sitagliptin|exenatide|alogliptin|saxagliptin|linagliptin|ertugliflozin|dapagliflozin|empagliflozin|canagliflozin|dulaglutide|semaglutide|liraglutide|lixisenatide|glucophage|glucotrol|amaryl|januvia|janumet|tradjenta|jardiance|farxiga|invokana|victoza|byetta"

LAB_MAP = {
    50809: ("glucose", "mg/dl"),
    50931: ("glucose", "mg/dl"),
    52027: ("glucose", "mg/dl"),
    52569: ("glucose", "mg/dl"),
    50852: ("hba1c", "%"),
    50912: ("creatinine", "mg/dl"),
    52024: ("creatinine", "mg/dl"),
    52546: ("creatinine", "mg/dl"),
    51006: ("bun", "mg/dl"),
    52647: ("bun", "mg/dl"),
    50862: ("albumin", "g/dl"),
    53085: ("albumin", "g/dl"),
    50983: ("sodium", "meq/l"),
    50824: ("sodium", "meq/l"),
    52623: ("sodium", "meq/l"),
    50971: ("potassium", "meq/l"),
    50822: ("potassium", "meq/l"),
    52610: ("potassium", "meq/l"),
    50882: ("bicarbonate", "meq/l"),
    50803: ("bicarbonate", "meq/l"),
    50813: ("lactate", "mmol/l"),
    51222: ("hemoglobin", "g/dl"),
    50811: ("hemoglobin", "g/dl"),
    51300: ("white_cells", "k/ul"),
    51301: ("white_cells", "k/ul"),
    51265: ("platelets", "k/ul"),
    50861: ("alt", "iu/l"),
    50878: ("ast", "iu/l"),
    50885: ("bilirubin", "mg/dl"),
    51237: ("inr", ""),
    50960: ("magnesium", "mg/dl"),
    50970: ("phosphate", "mg/dl"),
}

# Retain only fields consumed by the hosp-only feature builder.
MEAS_SCHEMA = pa.schema(
    [
        ("subject_id", pa.int64()),
        ("hadm_id", pa.int64()),
        ("charttime", pa.timestamp("ns")),
        ("storetime", pa.timestamp("ns")),
        ("concept", pa.string()),
        ("value", pa.float64()),
    ]
)
MED_SCHEMA = pa.schema(
    [
        ("subject_id", pa.int64()),
        ("hadm_id", pa.int64()),
        ("charttime", pa.timestamp("ns")),
        ("storetime", pa.timestamp("ns")),
        ("mask", pa.int32()),
        ("drughash", pa.uint64()),
        ("insulin", pa.bool_()),
        ("hyperkalemia", pa.bool_()),
        ("source", pa.string()),
        ("medication", pa.string()),
    ]
)

## Read diagnoses and treatment evidence

Stream diagnoses/prescriptions to define the retrospective T2DM population. Current-admission discharge diagnoses and all-record phenotype evidence are excluded from predictors.

In [ ]:
class Shards:
    def __init__(self, name, schema):
        self.folder = OUT / name
        self.folder.mkdir(parents=True, exist_ok=True)
        self.schema = schema
        self.writers = [
            pq.ParquetWriter(
                self.folder / f"part_{i:02d}.parquet", schema, compression="snappy"
            )
            for i in range(N)
        ]
        self.rows = 0

    def write(self, frame):
        if frame.empty:
            return
        frame = frame[self.schema.names].copy()
        frame["hadm_id"] = frame.hadm_id.astype("int64")
        for part, g in frame.groupby(frame.hadm_id % N):
            self.writers[int(part)].write_table(
                pa.Table.from_pandas(g, schema=self.schema, preserve_index=False, safe=True)
            )
        self.rows += len(frame)

    def close(self):
        for w in self.writers:
            w.close()


def foundation():
    marker = OUT / "foundation_complete.json"
    if marker.exists():
        return (
            pd.read_parquet(OUT / "patients.parquet"),
            pd.read_parquet(OUT / "admissions.parquet"),
            pd.read_parquet(OUT / "history.parquet"),
        )
    dictionary = pd.read_csv(ROOT / "d_icd_diagnoses.csv.gz", dtype={"icd_code": str})
    dictionary["key"] = dictionary.icd_version.astype(str) + "_" + dictionary.icd_code
    flags = {
        name: dict(
            zip(
                dictionary.key,
                dictionary.long_title.str.contains(pattern, case=False, na=False),
            )
        )
        for name, pattern in HISTORY_PATTERNS.items()
    }
    pieces = []
    for i, d in enumerate(
        pd.read_csv(
            ROOT / "diagnoses_icd.csv.gz",
            usecols=KEY + ["icd_code", "icd_version"],
            dtype={"icd_code": str},
            chunksize=CHUNK_SIZE,
        )
    ):
        code = d.icd_code.str.replace(".", "", regex=False).str.upper()
        nine = d.icd_version.eq(9)
        ten = d.icd_version.eq(10)
        temp = d[KEY].copy()
        temp["t2"] = (
            nine
            & code.str.startswith("250")
            & code.str.len().ge(5)
            & code.str[4].isin(["0", "2"])
            & ~code.isin(["25010", "25012"])
        ) | (ten & code.str.startswith("E11"))
        temp["t1"] = (
            nine
            & code.str.startswith("250")
            & code.str.len().ge(5)
            & code.str[4].isin(["1", "3"])
        ) | (ten & code.str.startswith("E10"))
        key = d.icd_version.astype(str) + "_" + d.icd_code
        for name, lookup in flags.items():
            temp["hx_" + name] = key.map(lookup).fillna(False).astype(bool)
        pieces.append(temp.groupby(KEY, sort=False).max().reset_index())
        log(f"DIAGNOSES: {(i+1)*CHUNK_SIZE:,} rows processed")
    history = pd.concat(pieces, ignore_index=True).groupby(KEY, sort=False).max().reset_index()
    patients = pd.read_csv(
        ROOT / "patients.csv.gz", usecols=["subject_id", "gender", "anchor_age", "anchor_year"]
    )
    totals = history.groupby("subject_id").agg(
        t2_admissions=("t2", "sum"), t1_any=("t1", "max")
    )
    patients = patients.merge(totals, on="subject_id", how="left")
    patients["t2_admissions"] = patients.t2_admissions.fillna(0).astype(int)
    patients["t1_any"] = patients.t1_any.fillna(False).astype(bool)
    admissions = pd.read_csv(
        ROOT / "admissions.csv.gz",
        usecols=KEY + ["admittime", "dischtime", "deathtime", "admission_type", "race"],
    )
    for c in ["admittime", "dischtime", "deathtime"]:
        admissions[c] = pd.to_datetime(admissions[c], errors="coerce", format="mixed")
    admissions["endtime"] = admissions[["dischtime", "deathtime"]].min(axis=1)
    for c in ["admission_type", "race"]:
        admissions[c] = admissions[c].astype("category")
    patients.to_parquet(OUT / "patients.parquet", index=False)
    admissions.to_parquet(OUT / "admissions.parquet", index=False)
    history.to_parquet(OUT / "history.parquet", index=False)
    marker.write_text(
        json.dumps(
            {
                "patients": len(patients),
                "admissions": len(admissions),
                "t2_coded_patients": int(patients.t2_admissions.gt(0).sum()),
            },
            indent=2,
        )
    )
    return patients, admissions, history


def prescriptions(patients):
    file = OUT / "noninsulin_prescriptions.parquet"
    if file.exists():
        return pd.read_parquet(file)
    found = []
    for i, d in enumerate(
        pd.read_csv(
            ROOT / "prescriptions.csv.gz",
            usecols=["subject_id", "drug", "starttime"],
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )
    ):
        d = d.loc[d.drug.str.contains(RX_PATTERN, case=False, na=False)].copy()
        d["starttime"] = pd.to_datetime(d.starttime, errors="coerce", format="mixed")
        found.append(d.groupby("subject_id").starttime.min())
        if i % 10 == 0:
            log(f"PRESCRIPTIONS: {(i+1)*CHUNK_SIZE:,} rows scanned")
    result = pd.concat(found).groupby(level=0).min().rename("first_noninsulin_rx").reset_index()
    result.to_parquet(file, index=False)
    log(f"Non-insulin Rx evidence: {len(result):,} patients")
    return result


def mask_for_names(names):
    unique = pd.Series(names.dropna().unique(), dtype=str)
    masks = np.zeros(len(unique), dtype=np.int32)
    for i, pattern in enumerate(MED_CLASSES.values()):
        masks |= (
            unique.str.contains(pattern, case=False, na=False).to_numpy(dtype=np.int32) << i
        )
    return dict(zip(unique, masks))

## Cache hosp laboratory and medication timelines

Laboratory glucose needs explicit mg/dL units or converted mmol/L units. eMAR requires `Administered`. Prescription orders support the phenotype but do not establish insulin exposure.

In [ ]:
def lab_scan(ids):
    marker = OUT / "labs_complete.json"
    if marker.exists():
        return
    sink = Shards("labs", MEAS_SCHEMA)
    a1c = {}
    columns = KEY + ["itemid", "charttime", "storetime", "valuenum", "valueuom"]
    for i, data in enumerate(
        pd.read_csv(
            ROOT / "labevents.csv.gz", usecols=columns, chunksize=CHUNK_SIZE, low_memory=False
        )
    ):
        data = data.loc[data.subject_id.isin(ids) & data.itemid.isin(LAB_MAP)].copy()
        if data.empty:
            continue
        data["concept"] = data.itemid.map(lambda item: LAB_MAP[item][0])
        data["value"] = pd.to_numeric(data.valuenum, errors="coerce")
        units = data.valueuom.fillna("").str.lower().str.replace(" ", "", regex=False)
        expected = data.itemid.map(lambda item: LAB_MAP[item][1])
        valid = units.eq(expected) | (data.concept.eq("hba1c") & units.isin(["", "%"]))
        mmol = data.concept.eq("glucose") & units.eq("mmol/l")
        data.loc[mmol, "value"] *= 18.0182
        data = data.loc[(valid | mmol) & np.isfinite(data.value) & data.value.gt(0)].copy()
        # Phenotype support may include outpatient HbA1c; features need an admission.
        for sid, value in (
            data.loc[data.concept.eq("hba1c")].groupby("subject_id").value.max().items()
        ):
            a1c[int(sid)] = max(a1c.get(int(sid), 0), float(value))
        data = data.loc[data.hadm_id.notna()].copy()
        for column in ["charttime", "storetime"]:
            data[column] = pd.to_datetime(data[column], errors="coerce", format="mixed")
        sink.write(data.loc[data.charttime.notna()])
        if i % 20 == 0:
            log(f"LAB SCAN: chunk {i+1}")
    sink.close()
    pd.Series(a1c, name="max_a1c").rename_axis("subject_id").reset_index().to_parquet(
        OUT / "max_a1c.parquet", index=False
    )
    marker.write_text(json.dumps({"cached_rows": sink.rows}))
    log(f"Labs cached: {sink.rows:,} readings")


def emar_scan(ids):
    marker = OUT / "emar_complete.json"
    if marker.exists():
        return
    sink = Shards("emar", MED_SCHEMA)
    exposures = []
    lookup = {}
    for i, d in enumerate(
        pd.read_csv(
            ROOT / "emar.csv.gz",
            usecols=KEY + ["charttime", "storetime", "medication", "event_txt"],
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )
    ):
        d = d.loc[
            d.subject_id.isin(ids) & d.hadm_id.notna() & d.event_txt.eq("Administered")
        ].copy()
        if not d.empty:
            for c in ["charttime", "storetime"]:
                d[c] = pd.to_datetime(d[c], errors="coerce", format="mixed")
            d["medication"] = (
                d.medication.fillna("").str.lower().str.replace(r"\s+", " ", regex=True)
            )
            missing = d.loc[~d.medication.isin(lookup), "medication"]
            lookup.update(mask_for_names(missing))
            d["mask"] = d.medication.map(lookup).astype(np.int32)
            d["drughash"] = pd.util.hash_pandas_object(d.medication, index=False).to_numpy(
                dtype=np.uint64
            )
            d["insulin"] = d.medication.str.contains("insulin", na=False)
            d["hyperkalemia"] = d.medication.str.contains("hyperkal", na=False)
            d["source"] = "emar"
            sink.write(d.loc[d.charttime.notna()])
            exposures.append(
                d.loc[
                    d.insulin,
                    KEY + ["charttime", "storetime", "source", "medication", "hyperkalemia"],
                ]
            )
        if i % 15 == 0:
            log(f"FULL EMAR SCAN: {(i+1)*CHUNK_SIZE:,} rows")
    sink.close()
    pd.concat(exposures, ignore_index=True).to_parquet(
        OUT / "emar_insulin.parquet", index=False
    )
    marker.write_text(json.dumps({"cached_rows": sink.rows}))
    log(f"eMAR cached: {sink.rows:,} administered records")

## Construct the supported T2DM population

Primary: ≥2 T2-coded admissions; or ≥1 plus non-insulin medication/HbA1c ≥6.5%; or medication plus HbA1c ≥6.5%. Exclude any Type 1 code. Keep broad-coded T2DM for sensitivity analysis.

In [ ]:
patients, admissions, history = foundation()
patients = patients.merge(prescriptions(patients), on="subject_id", how="left")
candidates = patients.loc[
    ~patients.t1_any & (patients.t2_admissions.gt(0) | patients.first_noninsulin_rx.notna())
].copy()
ids = set(candidates.subject_id)
# Sequential scans keep memory use predictable in a shared Colab runtime.
lab_scan(ids)
emar_scan(ids)
candidates = candidates.merge(
    pd.read_parquet(OUT / "max_a1c.parquet"), on="subject_id", how="left"
)
treatment = candidates.first_noninsulin_rx.notna()
elevated = candidates.max_a1c.ge(6.5)
candidates["phenotype_primary"] = (
    candidates.t2_admissions.ge(2)
    | (candidates.t2_admissions.ge(1) & (treatment | elevated))
    | (treatment & elevated)
)
candidates["phenotype_broad"] = candidates.t2_admissions.ge(1)
candidates = candidates.loc[candidates.phenotype_primary | candidates.phenotype_broad].copy()
print("Supported T2DM patients:", int(candidates.phenotype_primary.sum()))

## Select non-overlapping eMAR windows and prior history

Prediction time is max(actual administration, posting), with ≤60-minute delay. Select windows before outcome/baseline eligibility. Exclude explicitly named hyperkalemia index administrations later. All windows and admissions from a patient stay in one split.

In [ ]:
def assign_split(subject, old):
    if subject in old:
        return old[subject], False
    x = (
        int(hashlib.sha256(f"full-t2dm-20261027:{subject}".encode()).hexdigest()[:12], 16)
        / 16**12
    )
    return (
        "train"
        if x < 0.60
        else "validation" if x < 0.75 else "calibration" if x < 0.85 else "test"
    ), True


def select_episodes(exposures, admissions, patients):
    d = exposures.dropna(subset=["charttime", "storetime"]).drop_duplicates()
    d["t0"] = d[["charttime", "storetime"]].max(axis=1)
    d["lag_minutes"] = (d.t0 - d.charttime).dt.total_seconds() / 60
    d = (
        d.loc[d.lag_minutes.le(60)]
        .merge(admissions, on=KEY, validate="many_to_one")
        .merge(patients, on="subject_id", validate="many_to_one")
    )
    d["age"] = (d.anchor_age + d.charttime.dt.year - d.anchor_year).clip(upper=90)
    d = d.loc[d.age.ge(18) & d.charttime.ge(d.admittime) & d.t0.lt(d.endtime)].sort_values(
        ["subject_id", "t0", "source"]
    )
    indices = []
    last = {}
    for row in d[["subject_id", "t0"]].itertuples():
        if row.subject_id not in last or row.t0 >= last[row.subject_id]:
            indices.append(row.Index)
            last[row.subject_id] = row.t0 + 24 * H
    result = d.loc[indices].copy().rename(columns={"charttime": "admin_time"})
    result["full_followup"] = result.endtime.ge(result.t0 + 24 * H)
    result["episode_id"] = np.arange(len(result), dtype=np.int64)
    return result


def make_history_lookup(history, admissions):
    h = admissions[KEY + ["dischtime"]].merge(history, on=KEY, how="left")
    flags = [c for c in h if c.startswith("hx_")]
    h[flags] = h[flags].astype("boolean").fillna(False).astype(bool)
    h["t2"] = h.t2.astype("boolean").fillna(False).astype(bool)
    lookup = {}
    for sid, g in (
        h.dropna(subset=["dischtime"])
        .sort_values("dischtime")
        .groupby("subject_id", sort=False)
    ):
        values = g[flags].to_numpy(dtype=np.int8)
        lookup[int(sid)] = (
            g.dischtime.to_numpy(dtype="datetime64[ns]"),
            np.maximum.accumulate(values, axis=0),
            g.t2.to_numpy(dtype=bool),
            np.arange(1, len(g) + 1),
        )
    return lookup, flags


def monitoring(times, hours, max_gap):
    times = np.asarray(times, dtype=float)
    if len(times) == 0:
        return False
    # Readings are in (0,24]; exact right boundaries belong to preceding bin.
    bins = np.clip(np.ceil(times / hours).astype(int) - 1, 0, int(24 / hours) - 1)
    gaps = np.diff(np.r_[0.0, np.sort(times), 24.0])
    return len(np.unique(bins)) == int(24 / hours) and gaps.max() <= max_gap + 1e-9

## Construct strictly pre-exposure features and labels

Require a baseline glucose collected and available within six hours before actual administration, no existing low in the six-hour exclusion window and a full 24h stay. Both label classes need four six-hour monitoring bins and max gap ≤8h. Relaxed/strict alternatives are predefined. Unknown monitoring is never a negative.

In [ ]:
NH = np.timedelta64(1, "h")
NSH = 3_600_000_000_000


def compile_timeline(labs, medications):
    m = {c: labs[c].to_numpy() for c in ["charttime", "storetime", "value", "concept"]}
    d = {c: medications[c].to_numpy() for c in ["charttime", "storetime", "mask", "drughash"]}
    m["glucose"] = np.flatnonzero((m["concept"] == "glucose") & (m["value"] > 0))
    return m, d


def fast_episode(ep, cache, hist, flags):
    m, d = cache
    a = np.datetime64(ep.admin_time, "ns")
    t = np.datetime64(ep.t0, "ns")
    row = {
        name: getattr(ep, name)
        for name in [
            "episode_id",
            "subject_id",
            "hadm_id",
            "t0",
            "admin_time",
            "age",
            "gender",
            "race",
            "admission_type",
            "source",
            "hyperkalemia",
            "full_followup",
            "phenotype_primary",
            "phenotype_broad",
            "first_in_admission",
        ]
    }
    row["race"] = str(row["race"])
    row["admission_type"] = str(row["admission_type"])
    row["hours_since_admission"] = (ep.admin_time - ep.admittime).total_seconds() / 3600
    gi = m["glucose"]
    gt, gv, gs = m["charttime"][gi], m["value"][gi], m["storetime"][gi]
    low6 = bool(((gt >= t - 6 * NH) & (gt <= t) & (gv < 70)).any())
    low24 = bool(((gt >= t - 24 * NH) & (gt <= t) & (gv < 70)).any())
    # Both collection and posting precede the actual administration.
    pi = np.flatnonzero((gt < a) & (gs < a) & (gt >= a - 24 * NH))
    pi = pi[np.argsort(gt[pi], kind="quicksort")]
    if len(pi):
        pairs = np.rec.fromarrays([gt[pi].view("i8"), gv[pi]], names="t,v")
        _, first = np.unique(pairs, return_index=True)
        pi = pi[np.sort(first)]
    recent = bool((gt[pi] >= a - 6 * NH).any())
    row.update(baseline_available6h=recent, baseline_low6h=low6, baseline_low24h=low24)
    row.update(
        y70=np.nan,
        y54=np.nan,
        outcome_count=0,
        monitor_primary=False,
        monitor_relaxed=False,
        monitor_strict=False,
    )
    if not ep.full_followup or ep.hyperkalemia or not recent or low6:
        return row
    values = gv[pi]
    row.update(
        pre_glucose_last=float(values[-1]),
        pre_glucose_min=float(values.min()),
        pre_glucose_max=float(values.max()),
        pre_glucose_mean=float(values.mean()),
        pre_glucose_std=float(values.std(ddof=1)) if len(values) > 1 else np.nan,
        pre_glucose_count=float(len(values)),
        pre_glucose_age_hours=float((a - gt[pi[-1]]) / NH),
    )
    span = float((gt[pi[-1]] - gt[pi[0]]) / NH)
    row["pre_glucose_slope_per_hour"] = (
        float((values[-1] - values[0]) / span) if span > 0 else np.nan
    )
    future = (gt > t) & (gt <= t + 24 * NH)
    if not future.any():
        return row
    times, values = gt[future], gv[future]
    # Collapse duplicate/recheck monitoring within15min while retaining any low.
    _, inverse = np.unique(times.view("i8") // (NSH // 4), return_inverse=True)
    minimum = np.full(inverse.max() + 1, np.iinfo("int64").max, dtype="int64")
    np.minimum.at(minimum, inverse, times.view("i8"))
    hours = (minimum - t.view("i8")) / NSH
    row.update(
        y70=int(values.min() < 70), y54=int(values.min() < 54), outcome_count=len(minimum)
    )
    for kind, width, gap in [("primary", 6, 8), ("relaxed", 12, 12), ("strict", 4, 6)]:
        row["monitor_" + kind] = monitoring(hours, width, gap)
    if not any(row["monitor_" + kind] for kind in ["primary", "relaxed", "strict"]):
        return row
    # Latest available pre-dose laboratory observation for each concept.
    times = m["charttime"]
    available = (times < a) & (m["storetime"] < a) & (times >= a - 24 * NH)
    indices = np.flatnonzero(available)
    indices = indices[np.argsort(times[indices], kind="quicksort")]
    last = {str(m["concept"][i]): i for i in indices}
    for concept in sorted(set(x[0] for x in LAB_MAP.values()) - {"glucose"}):
        i = last.get(concept)
        row["lab_" + concept + "_last"] = float(m["value"][i]) if i is not None else np.nan
        row["lab_" + concept + "_age_hours"] = (
            float((a - times[i]) / NH) if i is not None else np.nan
        )
    use = (d["charttime"] < a) & (d["storetime"] < a) & (d["charttime"] >= a - 24 * NH)
    row["med_unique_drugs24h"] = int(len(np.unique(d["drughash"][use])))
    masks = d["mask"][use].astype(np.int32)
    for i, name in enumerate(MED_CLASSES):
        row["med_" + name + "_records24h"] = int(((masks & (1 << i)) != 0).sum())
    # Earlier completed admissions only; current final diagnosis codes cannot leak.
    record = hist.get(int(ep.subject_id))
    prior = False
    if record is not None:
        dates, values, t2s, count = record
        idx = np.searchsorted(dates, a, side="left") - 1
        for j, name in enumerate(flags):
            row[name] = int(values[idx, j]) if idx >= 0 else 0
        row["hx_prior_admissions"] = int(count[idx]) if idx >= 0 else 0
        row["hx_days_since_last_discharge"] = (
            float((a - dates[idx]) / np.timedelta64(1, "D")) if idx >= 0 else np.nan
        )
        prior = bool(t2s[: idx + 1].any()) if idx >= 0 else False
    else:
        for name in flags:
            row[name] = 0
        row["hx_prior_admissions"] = 0
        row["hx_days_since_last_discharge"] = np.nan
    row["t2_known_prior_discharge"] = prior
    return row

## Assemble the cohort and patient allocation

Reuse per-partition checkpoints. New patients receive a deterministic 60/15/10/15 split. An optional earlier manifest preserves previous allocations; without it, reported historical metrics will not be reproduced.

In [ ]:
def build_rows(episodes, admissions, history):
    final = OUT / "all_episode_rows.parquet"
    if final.exists():
        return pd.read_parquet(final)
    relevant = set(episodes.subject_id)
    hist, flags = make_history_lookup(
        history.loc[history.subject_id.isin(relevant)],
        admissions.loc[admissions.subject_id.isin(relevant)],
    )
    rows = []
    rowfolder = OUT / "episode_rows"
    rowfolder.mkdir(exist_ok=True)
    for part in range(N):
        target = episodes.loc[episodes.hadm_id.mod(N).eq(part)]
        if target.empty:
            continue
        checkpoint = rowfolder / f"part_{part:02d}.parquet"
        if checkpoint.exists():
            rows.append(pd.read_parquet(checkpoint))
            continue
        hadms = set(target.hadm_id)
        measurements = pd.read_parquet(OUT / "labs" / f"part_{part:02d}.parquet")
        measurements = measurements.loc[
            measurements.hadm_id.isin(hadms) & measurements.value.gt(0)
        ]
        medications = pd.read_parquet(OUT / "emar" / f"part_{part:02d}.parquet")
        medications = medications.loc[medications.hadm_id.isin(hadms)]
        mlookup = {hadm: g for hadm, g in measurements.groupby("hadm_id", sort=False)}
        dlookup = {hadm: g for hadm, g in medications.groupby("hadm_id", sort=False)}
        empty_m = measurements.iloc[:0]
        empty_d = medications.iloc[:0]
        timeline = {
            hadm: compile_timeline(mlookup.get(hadm, empty_m), dlookup.get(hadm, empty_d))
            for hadm in hadms
        }
        partrows = [
            fast_episode(ep, timeline[ep.hadm_id], hist, flags) for ep in target.itertuples()
        ]
        partframe = pd.DataFrame(partrows)
        partframe.to_parquet(checkpoint, index=False)
        rows.append(partframe)
        log(f"EPISODES: shard {part + 1}/{N}; {sum(map(len, rows)):,} labeled/featured windows")
        del partrows, partframe
        del measurements, medications, mlookup, dlookup, timeline
    result = pd.concat(rows, ignore_index=True)
    old = {}
    if EXISTING_SPLITS is not None:
        prior = (
            pd.read_parquet(EXISTING_SPLITS)
            if EXISTING_SPLITS.suffix == ".parquet"
            else pd.read_csv(EXISTING_SPLITS)
        )
        if prior.groupby("subject_id")["split"].nunique().max() != 1:
            raise ValueError("Existing patient splits conflict.")
        old = prior.drop_duplicates("subject_id").set_index("subject_id")["split"].to_dict()
        if not set(old.values()) <= {"train", "validation", "calibration", "test"}:
            raise ValueError("Unknown split label in manifest.")
    allocation = {int(s): assign_split(int(s), old) for s in result.subject_id.unique()}
    result["split"] = result.subject_id.map(lambda s: allocation[int(s)][0])
    result["new_patient"] = result.subject_id.map(lambda s: allocation[int(s)][1])
    result.to_parquet(final, index=False)
    return result


def eligibility(
    rows, monitor="primary", phenotype="primary", suffix="", first_only=False, asof=False
):
    mask = (
        rows["phenotype_" + phenotype]
        & rows.full_followup
        & ~rows.hyperkalemia
        & rows.baseline_available6h
        & ~rows.baseline_low6h
        & rows["monitor_" + monitor + suffix]
        & rows["y70" + suffix].notna()
    )
    if first_only:
        mask &= rows.first_in_admission
    if asof:
        mask &= rows.t2_known_prior_discharge.astype("boolean").fillna(False).astype(bool)
    result = rows.loc[mask].copy()
    result["target"] = result["y70" + suffix].astype(int)
    return result

## Export the complete modeling handoff

There are six cohorts. The former source/unit sensitivities are redundant when all glucose already comes from the same hosp lab source, so they are omitted. No patient rows are displayed.

In [ ]:
exposures = pd.read_parquet(OUT / "emar_insulin.parquet")
exposures = exposures.loc[exposures.subject_id.isin(candidates.subject_id)]
if exposures.empty:
    raise ValueError("No eligible eMAR insulin records in the candidate population.")
episodes = select_episodes(exposures, admissions, candidates)
first = episodes.sort_values("t0").drop_duplicates("hadm_id").episode_id
episodes["first_in_admission"] = episodes.episode_id.isin(first)
assert episodes.groupby("subject_id").t0.diff().dropna().ge(pd.Timedelta(hours=24)).all()
rows = build_rows(episodes, admissions, history)
assert rows.groupby("subject_id")["split"].nunique().max() == 1
assert set(rows.source) == {"emar"}
configurations = {
    "primary": {},
    "relaxed_monitoring": {"monitor": "relaxed"},
    "strict_monitoring": {"monitor": "strict"},
    "first_per_admission": {"first_only": True},
    "broad_coded_T2": {"phenotype": "broad"},
    "prior_documented_T2": {"asof": True},
}
counts = []
for name, options in configurations.items():
    cohort = eligibility(rows, **options)
    cohort.to_parquet(READY_DIR / f"{name}_cohort.parquet", index=False)
    counts.append(
        {
            "cohort": name,
            "patients": cohort.subject_id.nunique(),
            "admissions": cohort.hadm_id.nunique(),
            "windows": len(cohort),
            "positive_windows": int(cohort.target.sum()),
            "positive_patients": cohort.loc[cohort.target.eq(1)].subject_id.nunique(),
        }
    )
rows[["subject_id", "split"]].drop_duplicates().to_parquet(
    READY_DIR / "patient_splits.parquet", index=False
)
pd.DataFrame(counts).to_csv(READY_DIR / "cohort_counts.csv", index=False)
(READY_DIR / "protocol.json").write_text(
    json.dumps(
        {
            **signature,
            "target": "documented lab glucose <70 mg/dL in next24h after documented eMAR insulin",
            "retrospective_phenotype": True,
            "existing_split_manifest_supplied": EXISTING_SPLITS is not None,
            "baseline_hours": 6,
            "predictor_window_hours": 24,
            "primary_monitoring": "four6h bins, maxgap8h, symmetric labels",
            "labs": LAB_MAP,
            "medication_classes": MED_CLASSES,
            "prior_diagnosis_groups": HISTORY_PATTERNS,
        },
        indent=2,
    )
)
display(pd.DataFrame(counts))
print("Ready for notebook 2:", READY_DIR)